# Matter power spectra of a batch

The entry point to DISCO-EB: the linear matter power spectrum $P(k)$ of 128 flat $\Lambda$CDM cosmologies, computed and timed.

One `jax.jit`-ed function makes the batch, solves the Einstein-Boltzmann system of every (cosmology, $k$) pair in one GPU launch with `solve_eb` and turns what it kept into $P(k)$ with `matter_power_spectrum`.

In [1]:
import time

import jax
import jax.numpy as jnp
import numpy as np

from discoeb import LCDM, matter_power_spectrum, solve_eb

N_COSMOLOGIES = 128
K = np.geomspace(1.0e-4, 1.0, 128)  # wave numbers, 1/Mpc

## The jitted function

A cosmology is an instance of a model class, here `LCDM`. Its parameters are physical densities, $\omega = \Omega h^2$, and `LCDM.from_density_parameters` makes one from $\Omega_m$, $\Omega_b$ and $H_0$ by finding the dark-energy density that closes the budget. The batch draws $\omega_b$, $\omega_c$ and $h$ uniformly within under a per cent of the Planck 2018 best fit. Every parameter is traced, so a new batch reuses the compiled function.

`solve_eb` integrates in the scale factor, so `a = 1` is today for every cosmology and nothing is computed before the solve. It keeps the density contrasts of the cold dark matter and the baryons (`clxc`, `clxb`) there. `matter_power_spectrum` is one cosmology's, so `jax.vmap` maps it over the batch.

In [2]:
@jax.jit
def matter_power(key: jax.Array) -> jax.Array:
    """``P(k)`` in Mpc^3 today, ``(N_COSMOLOGIES, len(K))``, of a batch about Planck 2018.

    The batch draws ``omega_b``, ``omega_c`` and ``h`` uniformly within under
    a per cent of the Planck 2018 best fit (flat, three massless neutrino
    flavours), each independently. Each ``key`` gives a different batch.
    """

    u = jax.random.uniform(key, (3, N_COSMOLOGIES), minval=-1.0, maxval=1.0)
    omega_b = 0.02238280 * (1.0 + 0.006 * u[0])
    omega_c = 0.1201075 * (1.0 + 0.008 * u[1])
    h = 0.6732117 * (1.0 + 0.004 * u[2])
    A_s, n_s, k_p = 2.100549e-9, 0.9660499, 0.05  # k_p: pivot scale of A_s, 1/Mpc
    cosmologies = [
        LCDM.from_density_parameters(
            Omegam=(omega_b[i] + omega_c[i]) / h[i] ** 2,
            Omegab=omega_b[i] / h[i] ** 2,
            H0=100.0 * h[i],
            A_s=A_s,
            n_s=n_s,
            k_p=k_p,
            Tcmb=2.7255,
            YHe=0.2454006,
            Neff=3.046,
        )
        for i in range(N_COSMOLOGIES)
    ]

    values = solve_eb(  # (cosmology, name, k, a)
        cosmologies,
        K,
        [1.0],  # the scale factor today
        keep=("clxc", "clxb"),
        # recombination ten times looser than the hierarchy: moves P(k) by
        # under 6e-4 and takes 150 steps per trajectory instead of 225
        thermo_weight=0.1,
    )
    clxc, clxb = jnp.unstack(values[..., -1], axis=1)  # each (cosmology, k)
    # one cosmology's P(k), mapped over the batch with K held
    return jax.vmap(matter_power_spectrum, in_axes=(None,))(
        K,
        clxc=clxc,
        clxb=clxb,
        omega_c=omega_c,
        omega_b=omega_b,
        A_s=jnp.full(N_COSMOLOGIES, A_s),
        n_s=jnp.full(N_COSMOLOGIES, n_s),
        k_p=jnp.full(N_COSMOLOGIES, k_p),
    )

## Timing

The first call traces the function and loads the GPU kernel, which modax compiles once (about 35 s) and then caches on disk. The second call runs on a batch not seen before, as in an inference loop.

In [3]:
def timed(key: jax.Array) -> tuple[np.ndarray, float]:
    start = time.perf_counter()
    pk = jax.block_until_ready(matter_power(key))
    return np.asarray(pk), time.perf_counter() - start


warm_up_key, key = jax.random.split(jax.random.key(0))
_, first = timed(warm_up_key)
pk, elapsed = timed(key)  # a batch not seen before

## The result

In [4]:
print(f"first call, with tracing and the kernel's compile or load: {first:.1f} s")
print(f"P(k) of {pk.shape[0]} new cosmologies x {pk.shape[1]} modes: {elapsed:.3f} s")
print()
print("the first cosmology of the batch:")
print(f"{'k [1/Mpc]':>12}  {'P(k) [Mpc^3]':>14}")
first_pk = pk[0]
for kv, pv in zip(K[::8], first_pk[::8]):
    print(f"{kv:12.4e}  {pv:14.6e}")
print(f"P(k) peaks at k = {K[np.argmax(first_pk)]:.4g} 1/Mpc")

first call, with tracing and the kernel's compile or load: 3.3 s
P(k) of 128 new cosmologies x 128 modes: 0.218 s

the first cosmology of the batch:
   k [1/Mpc]    P(k) [Mpc^3]
  1.0000e-04    2.017580e+03
  1.7864e-04    3.529697e+03
  3.1911e-04    6.160566e+03
  5.7004e-04    1.068529e+04
  1.0183e-03    1.827247e+04
  1.8190e-03    3.034614e+04
  3.2495e-03    4.766200e+04
  5.8047e-03    6.779938e+04
  1.0369e-02    8.092413e+04
  1.8523e-02    6.972426e+04
  3.3089e-02    4.182546e+04
  5.9109e-02    2.346745e+04
  1.0559e-01    9.555350e+03
  1.8862e-01    3.453464e+03
  3.3695e-01    1.062762e+03
  6.0190e-01    2.966106e+02
P(k) peaks at k = 0.01115 1/Mpc
